## KNN on titanic dataset

We'll use again a well known dataset so that you can focus on the method.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# 0) Preprocessing

In [ ]:
# Red data
filename = 'https://raw.githubusercontent.com/Navenkumar-Balasubramaniam/00-General/main/16%20Decision%20Tree/titanic.xlsx'
df = pd.read_excel(filename, 1) #it has two sheets, we load the 2nd one

We first need to remove useles variables and null values (or impute them) since KNN in scikit does not handle them

In [ ]:
df.drop(labels=['cabin', 'ticket'], axis=1, inplace=True)
df=df.dropna()

Split the data and scale it

In [ ]:
from sklearn.model_selection import train_test_split

X = df.loc[:, df.columns != 'survived'] # removal of target variable from explanatories
y = df['survived']

X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.5, stratify=y) # Changed percentage=0.5 to test_size=0.5

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

standard_scaler = StandardScaler()
X_train = pd.DataFrame(standard_scaler.fit_transform(X_train))
X_test = pd.DataFrame(standard_scaler.transform(X_test))
print(X_train.shape)
print(X_test.shape)

#1) Single KNN mode

We can start with a standard k = 5. The only way to optimize k through trial and error with train data.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_classifier = KNeighborsClassifier(n_neighbors=5) # we don´t know yet the optimal number of neighbours
knn_classifier.fit(X_train, Y_train)

# Predictions (on test set, obviously)
knn_predictions = knn_classifier.predict(X_test)

# Calculate error_rate and print it
error_rate = 1 - accuracy_score(Y_test, knn_predictions)
print("% Error (k=5): {0:.1%}".format(error_rate))

#2) KNN model with grid search

Let's use GridSearchCV to find an optimal nuber of k, and an appropriate method: weighted or uniform (not weighted)

In [ ]:
# Grid search
from sklearn.model_selection import GridSearchCV

parameters = {
    "n_neighbors": range(1, 51),
    "weights": ["uniform", "distance"]
}
gridsearch = GridSearchCV(KNeighborsClassifier(), parameters)
gridsearch.fit(X_train, Y_train)

What are the best parameters found in the training dataset?

In [ ]:
gridsearch.best_params_

Predict using the model with the best parameters.

In [ ]:
gridpredictions = gridsearch.predict(X_test) # predicted class labels (0 or 1)
error_rate = 1 - accuracy_score(Y_test, gridpredictions)
print("% Error (grid search): {0:.1%}".format(error_rate))

We now create the ROC curve

In [ ]:
from sklearn.metrics import roc_curve, accuracy_score, auc

# Get predicted probabilities for the positive class
pred_prob = gridsearch.predict_proba(X_test)
prob_1 = pred_prob[:, 1]

# Calculate and print accuracies
train_acc = round(gridsearch.score(X_train, Y_train) * 100, 2)
test_acc = round(gridsearch.score(X_test, Y_test) * 100, 2)
error_rate = round(100 - test_acc, 2)
print("Error (grid search): ", (error_rate), "%")
print("Train Accuracy score: ", train_acc, "%")
print("Test Accuracy score: ", test_acc, "%")

# Calculate FPR, TPR, and thresholds
fpr, tpr, thresholds = roc_curve(Y_test, prob_1)
roc_auc = auc(fpr, tpr) # Corrected: Use auc(fpr, tpr) instead of roc_auc_score(fpr, tpr)

# Find optimal threshold (best trade-off between TPR and FPR)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal Threshold: {optimal_threshold:.2f}")

# ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')

# Highlight optimal threshold
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', marker='o',
            label=f'Optimal Threshold = {optimal_threshold:.2f}')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

print(f"AUC: {roc_auc:.3f}")

And confusion matrix

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

print("Confusion Matrix")
conf_mat = confusion_matrix(Y_test, gridpredictions)

# Draw heatmap using confusion matrix
sns.heatmap(conf_mat, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('True labels')
plt.ylabel('Predicted label');

#3) KNN model with bagging classifier

KNN's variance tends to be high. A Bagging classifier is an ensemble meta-estimator that fits base classifiers each on random subsets of the original dataset and then aggregate their individual predictions (either by voting or by averaging) to form a final prediction. It thus reduces variance by introducing randomization and aggregating.

In [ ]:
best_k = gridsearch.best_params_["n_neighbors"]
best_weights = gridsearch.best_params_["weights"]

# this is the base-model we will "bag"
bagged_knn = KNeighborsClassifier(
    n_neighbors=best_k, weights=best_weights
)

We are using here BaggingClassifier with 100 estimators/trees.

In [ ]:
from sklearn.ensemble import BaggingClassifier

knnbagging_model = BaggingClassifier(bagged_knn, n_estimators=100)
knnbagging_model.fit(X_train, Y_train)

And evaluate results with ROC curve and confusion matrix

In [ ]:
test_preds_grid = knnbagging_model.predict(X_test)

predictions = knnbagging_model.predict(X_test)
pred_prob = knnbagging_model.predict_proba(X_test)
prob_1 = [p[1] for p in pred_prob]

# Calculate FPR, TPR, and thresholds
fpr, tpr, thresholds = roc_curve(Y_test, prob_1)
roc_auc = auc(fpr, tpr) # Now directly using the imported auc function

# Find optimal threshold (best trade-off between TPR and FPR)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]
print(f"Optimal Threshold: {optimal_threshold:.2f}")

train_acc = round(knnbagging_model.score(X_train,Y_train) * 100,2) #Train Accuracy score
test_acc = round(knnbagging_model.score(X_test,Y_test) * 100,2) #Test Accuracy score
print("Train Accuracy score: ", train_acc, "%")
print("Test Accuracy score: ", test_acc, "%")

# ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Random classification

# Highlight optimal threshold
plt.scatter(fpr[optimal_idx], tpr[optimal_idx], color='red', marker='o',
            label=f'Optimal Threshold = {optimal_threshold:.2f}')

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

print(f"AUC: {roc_auc:.3f}")

#### And finally, we can change the prediction threshold

In [ ]:
threshold = 0.33
customized_predictions = (knnbagging_model.predict_proba(X_test)[:,1] >= threshold).astype(int)

In [ ]:
print("Confusion Matrix")
conf_mat = confusion_matrix(Y_test, customized_predictions)

# Draw heatmap using confusion matrix
sns.heatmap(conf_mat, square=True, annot=True, fmt='d', cbar=False)
plt.xlabel('True labels')
plt.ylabel('Predicted label');

# 4) A custom distance function to implement local domain knowledge

If the neighbour is older than the new observation, duplicate its weight. This function doesn't make much sense in this case. It is only for iustrative purposes. They can be applied for example in recomendation systems to weight more observations that were more recent.

In [ ]:
from scipy.spatial.distance import euclidean

def custom_age_weighted_distance(x1, x2):
    # Age column index is 2 (pclass, sex, age, sibsp, parch, fare, embarked)
    age_idx = 2

    # Calculate standard Euclidean distance
    base_distance = euclidean(x1, x2)

    # If the neighbor (x2) is older than the query point (x1),
    # reduce its distance to give it more 'weight'.
    # 'Older' is interpreted as having a higher scaled age value.
    if x2[age_idx] > x1[age_idx]:
        return base_distance / 2  # Halve the distance to double its effective weight
    else:
        return base_distance

In [ ]:
# Grid search with custom metric
from sklearn.model_selection import GridSearchCV

parameters_custom = {
    "n_neighbors": range(1, 51),
    "weights": ["uniform", "distance"] # Can still explore weights even with custom metric
}

# Pass the custom distance function to the 'metric' parameter
gridsearch_custom = GridSearchCV(KNeighborsClassifier(metric=custom_age_weighted_distance), parameters_custom)
gridsearch_custom.fit(X_train, Y_train)